# SVD-Based Recommendation System

In [111]:
import pandas as pd
import numpy as np
import re
from sklearn.decomposition import TruncatedSVD
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

In [112]:
# Only adjust the CLUSTER, WEIGHT_SVD, WEIGHT_DNN, default_weights, DATA_PATH, PLAYLIST_FILE, TRACKS_FILE

# ========== GLOBAL CONFIGURATION ==========
CLUSTER = None
WEIGHT_SVD = 0.5
WEIGHT_DNN = 0.5
DATA_PATH = "/Users/xavierhua/Documents/GitHub/spotifynd/phase5_model_development"
PLAYLIST_FILE = f"{DATA_PATH}/playlist_final_final.csv"
TRACKS_FILE = f"{DATA_PATH}/tracks_new.csv"
NUM_PLAYLISTS = 1000
K_EVAL = 50

# ========== SVD ==========
LATENT_DIM = 100

# ========== DNN ==========
# Default Feature Weights
default_weights = {
    'popularity': 0.2,
    'era': 0.2,
    'length': 0.2,
    'sentiment': 0.2,
    'genre': 0.2
}
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Model Architecture
INPUT_SIZE = None  # will be determined later based on the dataset
HIDDEN_SIZES = [512, 256, 128]
DROPOUT_RATE = 0.2
ACTIVATION = nn.ReLU

# Training Parameters
BATCH_SIZE = 512
EPOCHS = 20
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
LR_STEP_SIZE = 5
LR_GAMMA = 0.5
NEGATIVE_SAMPLE_RATIO = 10  # n_neg in PlaylistTrackDataset

# Loss multiplier
POS_WEIGHT_MULTIPLIER = 1.5

Using device: cpu


In [113]:
###################################
# 1. DATA LOADING & PREPROCESSING (SVD)
###################################
def load_data():
    playlists = pd.read_csv(PLAYLIST_FILE, engine='python', on_bad_lines='skip')
    tracks = pd.read_csv(TRACKS_FILE)
    if CLUSTER:
        playlists = playlists[playlists['cluster'] == CLUSTER].reset_index(drop=True)[:NUM_PLAYLISTS]
    else:
        playlists = playlists[:NUM_PLAYLISTS]
    return playlists, tracks

def clean_centroid_string(s):
    """Converts a centroid string into a list of floats."""
    if isinstance(s, str):
        s = re.sub(r'[\[\]]', '', s).strip()
        return [float(x) for x in s.split()]
    return s

def parse_track_list(s):
    """Extracts integers from a string and returns them as a list."""
    return [int(x) for x in re.findall(r'\d+', s)]

def preprocess_playlists(df):
    """Preprocesses the playlist DataFrame by cleaning centroids and parsing track lists."""
    df = df.copy()
    for col in ['sentiment_centroid', 'genre_centroid']:
        if col in df.columns:
            df[col] = df[col].apply(clean_centroid_string)
    for col in ['track_idx_list', 'tracks_to_predict']:
        if col in df.columns:
            df[col] = df[col].apply(parse_track_list)
    return df

In [114]:
###################################
# 2. DATA SPLITTING & INTERACTION MATRIX (SVD)
###################################
def split_data(playlists):
    """
    Splits playlists into train, validation, and test sets
    based on the 'dataset_type' column.
    """
    train = playlists[playlists['dataset_type'].isin(['train', 'val'])].reset_index(drop=True)
    val = playlists[playlists['dataset_type'] == 'val'].reset_index(drop=True)
    test = playlists[playlists['dataset_type'] == 'test'].reset_index(drop=True)
    final = playlists[playlists['dataset_type'] == 'final'].reset_index(drop=True)
    return train, val, test, final

def build_track_mapping(tracks_df):
    """Creates a sorted list of all available tracks and a mapping to column indices."""
    unique_tracks = sorted(tracks_df['track_idx'].unique().tolist())
    track_to_col = {track: idx for idx, track in enumerate(unique_tracks)}
    return unique_tracks, track_to_col

def build_interaction_matrix(playlists_subset, track_to_col, track_col='track_idx_list'):
    """Builds a binary interaction matrix for the given subset of playlists."""
    n = len(playlists_subset)
    m = len(track_to_col)
    matrix = np.zeros((n, m), dtype=np.int32)
    for i, row in playlists_subset.iterrows():
        for track in row[track_col]:
            if track in track_to_col:
                matrix[i, track_to_col[track]] = 1
    return matrix

def create_track_feat_map_svd(tracks_df):
    """
    Creates a feature mapping for tracks using the full tracks DataFrame.
    """
    feat_cols = ['track_popularity',
                 'Early Years', 'Classic Era', 'Golden Era', '2000s', 'Modern Era',
                 'Short', 'Medium', 'Long',
                 'joy', 'calm', 'sadness', 'fear', 'energizing', 'dreamy',
                 'Instrumental / Ambient Sounds', 'Soft Acoustic / Classical', 'Orchestral / Soundtrack',
                 'Mid-tempo Pop / Indie', 'Upbeat Electronic / Dance', 'Slow & Melancholic (Sad Songs)',
                 'Experimental / Jazz Fusion', 'Lo-Fi / Chill Vibes']
    feat_map = {row['track_idx']: row[feat_cols].values for _, row in tracks_df.iterrows()}
    return feat_map

In [115]:
###################################
# 3. SVD MODEL TRAINING, PREDICTION & EVALUATION (SVD)
###################################
def train_svd_model(interaction_matrix_train):
    """Trains an SVD model on the training interaction matrix."""
    svd = TruncatedSVD(n_components=LATENT_DIM, random_state=42)
    U_train = svd.fit_transform(interaction_matrix_train)
    Sigma = svd.singular_values_
    VT = svd.components_
    sqrt_sigma = np.sqrt(Sigma)
    P_train = U_train * sqrt_sigma  # Playlist latent factors
    Q = (VT.T * sqrt_sigma)         # Track latent factors
    return svd, P_train, Q, sqrt_sigma

def fold_in_playlists(svd, interaction_matrix, sqrt_sigma):
    """Folds in new playlists into the SVD latent space."""
    return svd.transform(interaction_matrix) * sqrt_sigma

def predict_mf(playlist_latent, interaction_row, Q):
    """
    Predicts scores for all tracks for a given playlist using its latent factors.
    Existing tracks are excluded by setting their scores to -∞.
    """
    scores = playlist_latent.dot(Q.T)
    existing = np.where(interaction_row > 0)[0]
    scores[existing] = -np.inf
    return scores

def predict_mf_wrapper(playlist_idx, interaction_matrix, P_matrix, Q):
    """Wrapper to predict scores for a playlist by its index."""
    return predict_mf(P_matrix[playlist_idx], interaction_matrix[playlist_idx], Q)

def get_all_svd_predictions(interaction_matrix, P_matrix, predict_func, Q, test_playlists):
    """
    Returns a dictionary mapping the original playlist IDs (from test_playlists) 
    to the predicted score vector for each playlist.
    """
    predictions = {}
    n = interaction_matrix.shape[0]
    for i in range(n):
        # Get the original playlist ID
        playlist_id = test_playlists.iloc[i]['playlist_idx']
        predictions[playlist_id] = predict_func(i, interaction_matrix, P_matrix, Q)
    return predictions

def compute_metrics_for_playlist(predicted_scores, test_indices, k=10):
    """
    Computes Hit@k, MRR, and MAP for a given playlist.
    """
    ranked_indices = np.argsort(-predicted_scores)
    top_k = ranked_indices[:k]
    hit = 1 if any(t in top_k for t in test_indices) else 0
    precisions = []
    num_hits = 0
    mrr = 0.0
    for rank_idx, track_idx in enumerate(ranked_indices[:k]):
        if track_idx in test_indices:
            num_hits += 1
            precisions.append(num_hits / (rank_idx + 1))
            if mrr == 0.0:
                mrr = 1.0 / (rank_idx + 1)
    ap = np.mean(precisions) if precisions else 0.0
    return hit, mrr, ap

def build_ground_truth(playlists_subset, track_to_col, track_col='tracks_to_predict'):
    """
    Builds a dictionary mapping each playlist (using its original 'playlist_idx')
    to a list of ground truth track indices.
    """
    test_items = {}
    for _, row in playlists_subset.iterrows():
        pid = row['playlist_idx']
        test_items[pid] = [track_to_col[t] for t in row[track_col] if t in track_to_col]
    return test_items

def evaluate_model_svd(interaction_matrix, ground_truth, P_matrix, predict_func, Q, playlistid_to_index, k=50):
    """
    Evaluates the SVD model using ground truth keyed by original playlist IDs.
    It returns only Hit@K, MRR, and MAP@K.
    """
    hit_total, mrr_total, ap_total = 0, 0, 0
    n = len(ground_truth)
    for pid, true_indices in ground_truth.items():
        if pid not in playlistid_to_index:
            continue  # skip if the playlist ID is not found in the test data
        idx = playlistid_to_index[pid]
        predicted_scores = predict_func(idx, interaction_matrix, P_matrix, Q)
        hit, mrr, ap = compute_metrics_for_playlist(predicted_scores, true_indices, k)
        hit_total += hit
        mrr_total += mrr
        ap_total += ap
    return {
        'Hit@K': hit_total / n,
        'MRR': mrr_total / n,
        'MAP@K': ap_total / n
    }

# TRAIN

In [116]:
# Load and preprocess data
playlist_raw, tracks = load_data()
playlists = preprocess_playlists(playlist_raw)

# Split data into train, validation, test, and final sets
train_playlists, val_playlists, test_playlists, final_playlists = split_data(playlists)

# Build track mapping (based on training data) and interaction matrices
unique_tracks, track_to_col = build_track_mapping(tracks)
print("Training playlists:", len(train_playlists))
print("Unique tracks (from train):", len(unique_tracks))

interaction_matrix_train = build_interaction_matrix(train_playlists, track_to_col)
interaction_matrix_val = build_interaction_matrix(val_playlists, track_to_col)
interaction_matrix_test = build_interaction_matrix(test_playlists, track_to_col)
interaction_matrix_final = build_interaction_matrix(final_playlists, track_to_col)
print("Training interaction matrix shape:", interaction_matrix_train.shape)
print("Validation interaction matrix shape:", interaction_matrix_val.shape)
print("Test interaction matrix shape:", interaction_matrix_test.shape)
print("Final interaction matrix shape:", interaction_matrix_final.shape)

track_feat_map = create_track_feat_map_svd(tracks)

# Train SVD model on training set
svd, P_train, Q, sqrt_sigma = train_svd_model(interaction_matrix_train)

Training playlists: 763
Unique tracks (from train): 250426
Training interaction matrix shape: (763, 250426)
Validation interaction matrix shape: (98, 250426)
Test interaction matrix shape: (104, 250426)
Final interaction matrix shape: (133, 250426)


# TEST

In [117]:
# Fold in test playlists
P_test = fold_in_playlists(svd, interaction_matrix_test, sqrt_sigma)
# Build ground truth for test
ground_truth_test = build_ground_truth(test_playlists, track_to_col)
# Build mapping from original playlist IDs to row indices for the test set
playlistid_to_index_test = {row['playlist_idx']: i for i, row in test_playlists.iterrows()}
# Evaluate on test set
svd_metrics_test = evaluate_model_svd(interaction_matrix_test, ground_truth_test, P_test, predict_mf_wrapper, Q, playlistid_to_index_test, k=K_EVAL)
# print("SVD Evaluation on Test:")
# print(f"Hit@{K_EVAL}: {svd_metrics_test['Hit@K']:.4f}")
# print(f"MRR:         {svd_metrics_test['MRR']:.4f}")
# print(f"MAP@{K_EVAL}: {svd_metrics_test['MAP@K']:.4f}")

# Convert test recommendations to DataFrame
index_to_track = {v: k for k, v in track_to_col.items()}
test_recommendations = []
n_test = interaction_matrix_test.shape[0]
for i in range(n_test):
    scores = predict_mf_wrapper(i, interaction_matrix_test, P_test, Q)
    ranked_indices = np.argsort(-scores)
    top_indices = ranked_indices[:K_EVAL]
    recommended_tracks = [index_to_track[idx] for idx in top_indices if idx in index_to_track]
    playlist_id = test_playlists.iloc[i]['playlist_idx']
    test_recommendations.append({'playlist_idx': playlist_id, 'recommended_tracks': recommended_tracks})
svd_rec_df = pd.DataFrame(test_recommendations)

In [118]:
svd_rec_df['playlist_idx'].duplicated().sum()

np.int64(0)

# DNN-Based Recommendation System

In [119]:
###################################
# 1. DATA LOADING & PREPROCESSING (DNN)
###################################
# def load_data():
#     playlists = pd.read_csv(PLAYLIST_FILE, engine='python', on_bad_lines='skip')
#     tracks = pd.read_csv(TRACKS_FILE)
#     if CLUSTER:
#         playlists = playlists[playlists['cluster'] == CLUSTER].reset_index(drop=True)[:NUM_PLAYLISTS]
#     else:
#         playlists = playlists[:NUM_PLAYLISTS]
#     return playlists, tracks

# def split_data(playlists):
#     """
#     Splits playlists into train, validation, test, and final sets based on the 'dataset_type' column.
#     """
#     train = playlists[playlists['dataset_type'] == 'train'].reset_index(drop=True)
#     val = playlists[playlists['dataset_type'] == 'val'].reset_index(drop=True)
#     test = playlists[playlists['dataset_type'] == 'test'].reset_index(drop=True)
#     final = playlists[playlists['dataset_type'] == 'final'].reset_index(drop=True)
#     return train, val, test, final

def preprocess_playlist(playlist):
    # Convert centroids from strings to numpy arrays and unpack them
    playlist['sentiment_centroid'] = playlist['sentiment_centroid'].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))
    playlist['genre_centroid'] = playlist['genre_centroid'].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))
    
    sent_df = playlist['sentiment_centroid'].apply(pd.Series)
    sent_df.columns = [f'sent{i+1}' for i in range(sent_df.shape[1])]
    genre_df = playlist['genre_centroid'].apply(pd.Series)
    genre_df.columns = [f'genre{i+1}' for i in range(genre_df.shape[1])]
    
    playlist = pd.concat([playlist.drop(columns=['sentiment_centroid', 'genre_centroid']), sent_df, genre_df], axis=1)
    
    def convert_string_array_to_list(s):
        if isinstance(s, str):
            return [int(x) for x in re.findall(r'\d+', s)]
        return []
    
    playlist['track_idx_list'] = playlist['track_idx_list'].apply(convert_string_array_to_list)
    playlist['tracks_to_predict'] = playlist['tracks_to_predict'].apply(convert_string_array_to_list)
    
    cols = ['playlist_idx', 'dataset_type', 'track_idx_list', 'tracks_to_predict', 'cluster',
            'popularity_mean', 'era_early_years_proportion', 'era_classic_era_proportion', 'era_golden_era_proportion',
            'era_2000s_proportion', 'era_modern_era_proportion', 'length_short_proportion', 'length_medium_proportion',
            'length_long_proportion', 'sent1', 'sent2', 'sent3', 'sent4', 'sent5', 'sent6',
            'genre1', 'genre2', 'genre3', 'genre4', 'genre5', 'genre6', 'genre7', 'genre8']
    return playlist[cols]

# Load and preprocess
playlist_raw, tracks = load_data()
playlist = preprocess_playlist(playlist_raw)
train_playlists_dnn, val_playlists_dnn, test_playlists_dnn, final_playlists_dnn = split_data(playlist)

# Define playlist feature columns and create scalers
playlist_popularity_cols = ['popularity_mean']
playlist_era_cols = ['era_early_years_proportion', 'era_classic_era_proportion', 'era_golden_era_proportion',
                       'era_2000s_proportion', 'era_modern_era_proportion']
playlist_length_cols = ['length_short_proportion', 'length_medium_proportion', 'length_long_proportion']
playlist_sentiment_cols = ['sent1', 'sent2', 'sent3', 'sent4', 'sent5', 'sent6']
playlist_genre_cols = ['genre1', 'genre2', 'genre3', 'genre4', 'genre5', 'genre6', 'genre7', 'genre8']

playlist_scalers = {
    'pop_playlist': StandardScaler(),
    'era_playlist': StandardScaler(),
    'len_playlist': StandardScaler(),
    'sent_playlist': StandardScaler(),
    'genre_playlist': StandardScaler()
}

playlist_scalers['pop_playlist'].fit(train_playlists_dnn[playlist_popularity_cols])
playlist_scalers['era_playlist'].fit(train_playlists_dnn[playlist_era_cols])
playlist_scalers['len_playlist'].fit(train_playlists_dnn[playlist_length_cols])
playlist_scalers['sent_playlist'].fit(train_playlists_dnn[playlist_sentiment_cols])
playlist_scalers['genre_playlist'].fit(train_playlists_dnn[playlist_genre_cols])

def normalize_playlist_features(df):
    df[playlist_popularity_cols] = playlist_scalers['pop_playlist'].transform(df[playlist_popularity_cols])
    df[playlist_era_cols] = playlist_scalers['era_playlist'].transform(df[playlist_era_cols])
    df[playlist_length_cols] = playlist_scalers['len_playlist'].transform(df[playlist_length_cols])
    df[playlist_sentiment_cols] = playlist_scalers['sent_playlist'].transform(df[playlist_sentiment_cols])
    df[playlist_genre_cols] = playlist_scalers['genre_playlist'].transform(df[playlist_genre_cols])
    return df

train_playlists_dnn = normalize_playlist_features(train_playlists_dnn.copy())
val_playlists_dnn = normalize_playlist_features(val_playlists_dnn.copy())
test_playlists_dnn = normalize_playlist_features(test_playlists_dnn.copy())

def create_playlist_feat_map(df):
    feat_cols = playlist_popularity_cols + playlist_era_cols + playlist_length_cols + playlist_sentiment_cols + playlist_genre_cols
    feat_map = {row['playlist_idx']: row[feat_cols].values for _, row in df.iterrows()}
    return feat_map

playlist_feat_map_train = create_playlist_feat_map(train_playlists_dnn)
playlist_feat_map_val = create_playlist_feat_map(val_playlists_dnn)
playlist_feat_map_test = create_playlist_feat_map(test_playlists_dnn)

# For tracks in DNN, define feature columns and scalers
popularity_cols = ['track_popularity']
era_cols = ['Early Years', 'Classic Era', 'Golden Era', '2000s', 'Modern Era']
length_cols = ['Short', 'Medium', 'Long']
sentiment_cols = ['joy', 'calm', 'sadness', 'fear', 'energizing', 'dreamy']
genre_cols = ['Instrumental / Ambient Sounds', 'Soft Acoustic / Classical', 'Orchestral / Soundtrack',
              'Mid-tempo Pop / Indie', 'Upbeat Electronic / Dance', 'Slow & Melancholic (Sad Songs)',
              'Experimental / Jazz Fusion', 'Lo-Fi / Chill Vibes']

track_scalers = {
    'pop_track': StandardScaler(),
    'era_track': StandardScaler(),
    'len_track': StandardScaler(),
    'sent_track': StandardScaler(),
    'genre_track': StandardScaler()
}

track_scalers['pop_track'].fit(tracks[popularity_cols])
track_scalers['era_track'].fit(tracks[era_cols])
track_scalers['len_track'].fit(tracks[length_cols])
track_scalers['sent_track'].fit(tracks[sentiment_cols])
track_scalers['genre_track'].fit(tracks[genre_cols])

def normalize_track_features(df):
    df[popularity_cols] = track_scalers['pop_track'].transform(df[popularity_cols])
    df[era_cols] = track_scalers['era_track'].transform(df[era_cols])
    df[length_cols] = track_scalers['len_track'].transform(df[length_cols])
    df[sentiment_cols] = track_scalers['sent_track'].transform(df[sentiment_cols])
    df[genre_cols] = track_scalers['genre_track'].transform(df[genre_cols])
    return df

tracks = normalize_track_features(tracks.copy())

def create_track_feat_map(df):
    feat_cols = popularity_cols + era_cols + length_cols + sentiment_cols + genre_cols
    feat_map = {int(row['track_idx']): row[feat_cols].values for _, row in df.iterrows()}
    return feat_map

track_feat_map = create_track_feat_map(tracks)

def apply_weights(features, weights):
    split_sizes = [1, 5, 3, 6, 8]
    chunks = np.split(features, np.cumsum(split_sizes)[:-1])
    return np.concatenate([chunk * weights[key] for chunk, key in zip(chunks, weights)])

In [120]:
###################################
# 2. MODEL & DATASET DEFINITION (DNN)
###################################
class PlaylistTrackDataset(Dataset):
    def __init__(self, playlist_df, feat_map, track_map, n_neg=NEGATIVE_SAMPLE_RATIO, cluster_weights=None):
        self.samples = []
        self.track_map = track_map
        all_tids = list(track_map.keys())
        cluster_weights = cluster_weights or {}
        for _, row in playlist_df.iterrows():
            pid, cluster = row['playlist_idx'], row['cluster']
            pos_tracks = row['tracks_to_predict']
            if not pos_tracks:  # skip if no positive tracks
                continue
            p_feat = feat_map[pid]
            weights = cluster_weights.get(cluster, default_weights)
            p_feat_w = apply_weights(p_feat, weights)
            for tid in pos_tracks:
                if tid in track_map:
                    self.samples.append((p_feat_w, track_map[tid], 1))
            negs = np.random.choice(list(set(all_tids) - set(pos_tracks)), 
                                    min(len(pos_tracks) * n_neg, len(all_tids)), replace=False)
            for tid in negs:
                self.samples.append((p_feat_w, track_map[tid], 0))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        p, t, y = self.samples[idx]
        p = np.array(p, dtype=np.float32).flatten()
        t = np.array(t, dtype=np.float32).flatten()
        return torch.tensor(np.concatenate([p, t]), dtype=torch.float32), torch.tensor([y], dtype=torch.float32)

class DNNRecommender(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        layers = []
        prev_size = input_size
        for hidden in HIDDEN_SIZES:
            layers.append(nn.Linear(prev_size, hidden))
            layers.append(ACTIVATION())
            layers.append(nn.BatchNorm1d(hidden))
            layers.append(nn.Dropout(DROPOUT_RATE))
            prev_size = hidden
        layers.append(nn.Linear(prev_size, 1))
        self.model = nn.Sequential(*layers)
    def forward(self, x):
        return self.model(x).squeeze()

# TRAIN

In [121]:
# Create playlist feature maps for training and validation
playlist_feat_map_train = create_playlist_feat_map(train_playlists_dnn)
playlist_feat_map_val = create_playlist_feat_map(val_playlists_dnn)

# Create the dataset and DataLoader for training
train_dataset = PlaylistTrackDataset(train_playlists_dnn, playlist_feat_map_train, track_feat_map)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Initialize model, optimizer, and scheduler
model = DNNRecommender(input_size=len(next(iter(train_dataset))[0])).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP_SIZE, gamma=LR_GAMMA)

# Compute pos_weight for loss function
pos = sum(1 for _, _, l in train_dataset.samples if l == 1)
neg = sum(1 for _, _, l in train_dataset.samples if l == 0)
pos_weight_val = (neg/pos * POS_WEIGHT_MULTIPLIER)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_val).to(device))

# Training loop
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb).view(-1), yb.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 214.4052
Epoch 2, Loss: 191.9205
Epoch 3, Loss: 182.8362
Epoch 4, Loss: 177.4513
Epoch 5, Loss: 172.9926
Epoch 6, Loss: 165.5125
Epoch 7, Loss: 162.8374
Epoch 8, Loss: 161.1850
Epoch 9, Loss: 158.4439
Epoch 10, Loss: 156.6031
Epoch 11, Loss: 150.8906
Epoch 12, Loss: 149.7159
Epoch 13, Loss: 148.1850
Epoch 14, Loss: 146.0249
Epoch 15, Loss: 145.5484
Epoch 16, Loss: 141.7539
Epoch 17, Loss: 140.1069
Epoch 18, Loss: 139.9434
Epoch 19, Loss: 140.7682
Epoch 20, Loss: 138.6796


# TEST

In [122]:
def predict_dnn(playlist_idx, feat_map, track_feat_map):
    p_feat = feat_map[playlist_idx]
    p_feat_w = apply_weights(p_feat, default_weights)
    all_track_ids = list(track_feat_map.keys())
    p_feat_tensor = torch.tensor(np.array(p_feat_w, dtype=np.float32)).repeat(len(all_track_ids), 1).to(device)
    t_feats = np.array([track_feat_map[tid] for tid in all_track_ids], dtype=np.float32)
    t_tensor = torch.tensor(t_feats, dtype=torch.float32).to(device)
    inputs = torch.cat([p_feat_tensor, t_tensor], dim=1)
    model.eval()
    with torch.no_grad():
        scores = torch.sigmoid(model(inputs)).cpu().numpy().flatten()
    return scores

def get_all_dnn_predictions(feat_map, track_feat_map, test_playlists):
    predictions = {}
    for i, row in test_playlists.iterrows():
        playlist_id = row['playlist_idx']
        predictions[playlist_id] = predict_dnn(playlist_id, feat_map, track_feat_map)
    return predictions

def compute_metrics_for_playlist(predicted_tracks, true_tracks, k):
    top_k = predicted_tracks[:k]
    hit = int(any(t in top_k for t in true_tracks))
    precisions = []
    mrr = 0.0
    num_hits = 0
    for rank, track_id in enumerate(top_k):
        if track_id in true_tracks:
            num_hits += 1
            precisions.append(num_hits / (rank + 1))
            if mrr == 0.0:
                mrr = 1.0 / (rank + 1)
    ap = np.mean(precisions) if precisions else 0.0
    return hit, mrr, ap

def evaluate_model(model, test_playlists, playlist_feat_map, track_feat_map, k=K_EVAL, weights_by_cluster=None):
    model.eval()
    all_track_ids = list(track_feat_map.keys())
    total_hit, total_mrr, total_ap = 0, 0, 0
    n = len(test_playlists)
    with torch.no_grad():
        for _, row in test_playlists.iterrows():
            pid = row['playlist_idx']
            cluster = row['cluster']
            true_tracks = row['tracks_to_predict']
            seen_tracks = set(row['track_idx_list'])
            p_feat = playlist_feat_map[pid]
            weights = weights_by_cluster.get(cluster, default_weights) if weights_by_cluster else default_weights
            p_feat_w = apply_weights(p_feat, weights)
            p_feat_tensor = torch.tensor(np.array(p_feat_w, dtype=np.float32)).repeat(len(all_track_ids), 1)
            t_feats = np.array([track_feat_map[tid] for tid in all_track_ids], dtype=np.float32)
            t_tensor = torch.tensor(t_feats, dtype=torch.float32)
            inputs = torch.cat([p_feat_tensor, t_tensor], dim=1)
            scores = torch.sigmoid(model(inputs.to(device))).cpu().numpy().flatten()
            ranked_tracks = [tid for tid, _ in sorted(zip(all_track_ids, scores), key=lambda x: x[1], reverse=True)]
            ranked_unseen = [tid for tid in ranked_tracks if tid not in seen_tracks]
            hit, mrr, ap = compute_metrics_for_playlist(ranked_unseen, true_tracks, k)
            total_hit += hit
            total_mrr += mrr
            total_ap += ap
    return {
        'Hit@K': total_hit / n,
        'MRR': total_mrr / n,
        'MAP@K': total_ap / n
    }

# Use test set for evaluation
playlist_feat_map_test = create_playlist_feat_map(test_playlists_dnn)
# test_metrics = evaluate_model(model, test_playlists_dnn, playlist_feat_map_test, track_feat_map, k=K_EVAL)
# print("DNN Test Evaluation:")
# print(f"Hit@{K_EVAL}:             {test_metrics['Hit@K']:.4f}")
# print(f"MRR:                    {test_metrics['MRR']:.4f}")
# print(f"MAP@{K_EVAL}:             {test_metrics['MAP@K']:.4f}")

# Export recommendations from test set
test_recommendations = []
all_track_ids = list(track_feat_map.keys())
for _, row in test_playlists_dnn.iterrows():
    playlist_id = row['playlist_idx']
    scores = predict_dnn(playlist_id, playlist_feat_map_test, track_feat_map)
    ranked_indices = np.argsort(-scores)
    top_indices = ranked_indices[:K_EVAL]
    recommended_tracks = [all_track_ids[i] for i in top_indices if i < len(all_track_ids)]
    test_recommendations.append({'playlist_idx': playlist_id, 'recommended_tracks': recommended_tracks})
dnn_rec_df = pd.DataFrame(test_recommendations)
print(dnn_rec_df.head())

   playlist_idx                                 recommended_tracks
0             2  [177736, 3884, 138696, 159098, 17888, 205579, ...
1             7  [221989, 102256, 250469, 43730, 60873, 177774,...
2            10  [22708, 251421, 123705, 32222, 139812, 208590,...
3            17  [172474, 98751, 191269, 191316, 49570, 25208, ...
4            28  [149917, 160078, 136749, 95651, 18002, 167909,...


In [123]:
dnn_rec_df['playlist_idx'].duplicated().sum()

np.int64(0)

# Hybrid

In [124]:
# Get predictions on test using original playlist IDs as keys
svd_predictions = get_all_svd_predictions(interaction_matrix_test, P_test, predict_mf_wrapper, Q, test_playlists)
dnn_predictions = get_all_dnn_predictions(playlist_feat_map_test, track_feat_map, test_playlists_dnn)

# Ensure predictions from both models are aligned.
svd_track_ids = set(track_to_col.keys())
dnn_track_ids = set(track_feat_map.keys())
union_track_ids = sorted(svd_track_ids.union(dnn_track_ids))
union_track_to_index = {tid: idx for idx, tid in enumerate(union_track_ids)}
union_length = len(union_track_ids)

final_predictions = {}
all_playlist_ids = set(svd_predictions.keys()).union(set(dnn_predictions.keys()))
for pid in all_playlist_ids:
    final_score = np.zeros(union_length, dtype=np.float32)
    if pid in svd_predictions:
        svd_vec = svd_predictions[pid]
        for track, idx in track_to_col.items():
            union_idx = union_track_to_index.get(track)
            if union_idx is not None and idx < len(svd_vec):
                final_score[union_idx] += WEIGHT_SVD * svd_vec[idx]
    if pid in dnn_predictions:
        dnn_vec = dnn_predictions[pid]
        dnn_track_list = sorted(dnn_track_ids)
        dnn_map = {tid: idx for idx, tid in enumerate(dnn_track_list)}
        for track, dnn_idx in dnn_map.items():
            union_idx = union_track_to_index.get(track)
            if union_idx is not None and dnn_idx < len(dnn_vec):
                final_score[union_idx] += WEIGHT_DNN * dnn_vec[dnn_idx]
    final_predictions[pid] = final_score

def evaluate_final_predictions(final_preds, ground_truth, track_to_col, k=50):
    total_hit, total_mrr, total_ap = 0, 0, 0
    n = len(ground_truth)
    # Build a mapping from matrix indices to track IDs.
    index_to_track = {v: k for k, v in track_to_col.items()}
    for pid, true_indices in ground_truth.items():
        pred_scores = final_preds.get(pid)
        if pred_scores is None:
            continue
        ranked_indices = np.argsort(-pred_scores)
        top_k = ranked_indices[:k]
        # Compute Hit@K
        hit = 1 if any(t in true_indices for t in top_k) else 0
        total_hit += hit
        # Compute MRR and AP
        num_hits = 0
        precisions = []
        mrr = 0.0
        for rank, track_idx in enumerate(top_k):
            if track_idx in true_indices:
                num_hits += 1
                precisions.append(num_hits / (rank + 1))
                if mrr == 0:
                    mrr = 1 / (rank + 1)
        ap = np.mean(precisions) if precisions else 0.0
        total_ap += ap
        total_mrr += mrr
    return {
        'Hit@K': total_hit / n,
        'MRR': total_mrr / n,
        'MAP@K': total_ap / n
    }
playlist_clusters = {row['playlist_idx']: row['cluster'] for _, row in test_playlists.iterrows()}

# Evaluate overall final predictions
final_metrics = evaluate_final_predictions(final_predictions, ground_truth_test, track_to_col, k=K_EVAL)
print("Final Combined Model Evaluation:")
print(f"Hit@{K_EVAL}:             {final_metrics['Hit@K']:.4f}")
print(f"MRR:                {final_metrics['MRR']:.4f}")
print(f"MAP@{K_EVAL}:             {final_metrics['MAP@K']:.4f}")

# ===== CONVERT COMBINED RECOMMENDATIONS TO PANDAS =====
# Export overall combined model recommendations:
combined_recommendations = []
for pid, final_score in final_predictions.items():
    ranked_indices = np.argsort(-final_score)
    top_indices = ranked_indices[:K_EVAL]
    recommended_tracks = [union_track_ids[i] for i in top_indices if i < len(union_track_ids)]
    combined_recommendations.append({
        'playlist_idx': pid,
        'recommended_tracks': recommended_tracks
    })
combined_rec_df = pd.DataFrame(combined_recommendations)
print(combined_rec_df.head())

Final Combined Model Evaluation:
Hit@50:             0.5481
MRR:                0.1225
MAP@50:             0.0880
   playlist_idx                                 recommended_tracks
0             2  [97130, 66741, 3884, 130032, 112206, 171412, 7...
1           518  [237495, 64550, 86790, 216374, 203557, 116049,...
2             7  [191313, 216260, 174692, 79687, 186289, 251617...
3            10  [208248, 218307, 53989, 94251, 180201, 221104,...
4          1036  [86790, 221079, 235367, 129996, 49453, 58505, ...


In [125]:
combined_rec_df['playlist_idx'].duplicated().sum()

np.int64(0)

In [126]:
# Evaluate by cluster
def evaluate_final_predictions_by_cluster(final_preds, ground_truth, playlist_clusters, k=50):
    """
    Evaluates the final combined model predictions by grouping playlists by their cluster.
    Returns only Hit@K, MRR, and MAP@K for each cluster.
    Assumes that final_preds and ground_truth are keyed by the original playlist IDs.
    """
    cluster_metrics = {}
    clusters = {}
    # Group playlist IDs by their cluster (using original IDs)
    for pid, cluster in playlist_clusters.items():
        clusters.setdefault(cluster, []).append(pid)
        
    for cluster, pids in clusters.items():
        total_hit, total_mrr, total_ap = 0, 0, 0
        n = 0
        for pid in pids:
            if pid not in ground_truth or pid not in final_preds:
                continue
            n += 1
            true_indices = ground_truth[pid]
            pred_scores = final_preds[pid]
            ranked_indices = np.argsort(-pred_scores)
            top_k = ranked_indices[:k]
            # Compute Hit@K
            hit = 1 if any(t in true_indices for t in top_k) else 0
            total_hit += hit
            num_hits = 0
            precisions = []
            mrr = 0.0
            for rank, track_idx in enumerate(top_k):
                if track_idx in true_indices:
                    num_hits += 1
                    precisions.append(num_hits / (rank + 1))
                    if mrr == 0:
                        mrr = 1 / (rank + 1)
            ap = np.mean(precisions) if precisions else 0.0
            total_ap += ap
            total_mrr += mrr
        if n > 0:
            cluster_metrics[cluster] = {
                'Hit@K': total_hit / n,
                'MRR': total_mrr / n,
                'MAP@K': total_ap / n
            }
        else:
            cluster_metrics[cluster] = {}
    return cluster_metrics

cluster_metrics = evaluate_final_predictions_by_cluster(final_predictions, ground_truth_test, playlist_clusters, k=K_EVAL)

print("\nFinal Combined Model Evaluation by Cluster:")
for cluster, metrics in cluster_metrics.items():
    print(f"Cluster {cluster}:")
    print(f"  Hit@{K_EVAL}:             {metrics.get('Hit@K', 0):.4f}")
    print(f"  MRR:                {metrics.get('MRR', 0):.4f}")
    print(f"  MAP@{K_EVAL}:             {metrics.get('MAP@K', 0):.4f}")

# Export combined model recommendations by cluster:
combined_cluster_recommendations = []

for i, row in test_playlists.iterrows():
    if i in final_predictions:
        cluster = row['cluster']
        original_id = row['playlist_idx']
        final_score = final_predictions[i]
        ranked_indices = np.argsort(-final_score)
        top_indices = ranked_indices[:K_EVAL]
        recommended_tracks = [union_track_ids[j] for j in top_indices if j < len(union_track_ids)]
        combined_cluster_recommendations.append({
            'playlist_idx': original_id,
            'cluster': cluster,
            'recommended_tracks': recommended_tracks
        })
combined_cluster_df = pd.DataFrame(combined_cluster_recommendations)
print(combined_cluster_df.head())


Final Combined Model Evaluation by Cluster:
Cluster 3:
  Hit@50:             0.3478
  MRR:                0.0712
  MAP@50:             0.0391
Cluster 1:
  Hit@50:             0.6667
  MRR:                0.1249
  MAP@50:             0.0894
Cluster 0:
  Hit@50:             0.6667
  MRR:                0.2684
  MAP@50:             0.2133
Cluster 2:
  Hit@50:             0.5000
  MRR:                0.0859
  MAP@50:             0.0610
   playlist_idx  cluster                                 recommended_tracks
0            10        0  [97130, 66741, 3884, 130032, 112206, 171412, 7...
1            46        0  [191313, 216260, 174692, 79687, 186289, 251617...
2            60        3  [208248, 218307, 53989, 94251, 180201, 221104,...
3           165        1  [235367, 216374, 64550, 14975, 54047, 84826, 1...
4           334        1  [250306, 64550, 86790, 180201, 26659, 203253, ...


In [127]:
combined_cluster_df['playlist_idx'].duplicated().sum()

np.int64(0)

# Evaluation

In [128]:
playlist, tracks = load_data()

In [129]:
def convert_string_array_to_list(s):
    """Convert a string representation of an array into a list of integers."""
    if isinstance(s, str):  # Handle string case (where it's incorrectly stored)
        numbers = re.findall(r'\d+', s)  # Extract all numeric values
        return [int(x) for x in numbers]  # Convert to integers
    
    elif isinstance(s, np.ndarray):  # Handle NumPy array case
        return s.astype(int).tolist()
    
    elif isinstance(s, list):  # Handle lists with possible string numbers
        return [int(x) for x in s if str(x).isdigit()]
    
    return []  # Return empty list if the format is unexpected

# Apply function and debug output
playlist['tracks_to_predict'] = playlist['tracks_to_predict'].apply(lambda x: convert_string_array_to_list(x))

# Check if the transformation worked
print(playlist[['tracks_to_predict']].head(10))  # Print first 10 rows

                                   tracks_to_predict
0  [218708, 242974, 165272, 9418, 221860, 229224,...
1  [232845, 111887, 144160, 10663, 216321, 138869...
2  [42986, 156208, 73950, 54114, 134077, 214967, ...
3  [166268, 22122, 211269, 71335, 21853, 190702, ...
4  [209088, 186942, 33032, 27612, 33426, 201531, ...
5  [250306, 36356, 119562, 66146, 206650, 166690,...
6  [48557, 142203, 66610, 16213, 112277, 43008, 2...
7  [45655, 100810, 250258, 169756, 86872, 117869,...
8  [88490, 122326, 87495, 205574, 3345, 191269, 2...
9  [89301, 74022, 221104, 218307, 229795, 129045,...


In [130]:
merged = playlist.merge(combined_rec_df, on='playlist_idx', how='inner')
merged = merged[['playlist_idx', 'cluster', 'tracks_to_predict', 'recommended_tracks']]
merged

,playlist_idx,cluster,tracks_to_predict,recommended_tracks
0,2,3,"[232845, 111887, 144160, 10663, 216321, 138869...","[97130, 66741, 3884, 130032, 112206, 171412, 7..."
1,7,1,"[48557, 142203, 66610, 16213, 112277, 43008, 2...","[191313, 216260, 174692, 79687, 186289, 251617..."
2,10,0,"[89301, 74022, 221104, 218307, 229795, 129045,...","[208248, 218307, 53989, 94251, 180201, 221104,..."
3,17,2,"[131069, 102197, 241039, 46103, 41621, 213083,...","[235367, 216374, 64550, 14975, 54047, 84826, 1..."
4,28,2,"[166174, 63352, 37636, 75383, 195511, 131429, ...","[250306, 64550, 86790, 180201, 26659, 203253, ..."
...,...,...,...,...
99,1008,3,"[179922, 218673, 251321, 224126, 76949, 28501,...","[219637, 34319, 216260, 229551, 251617, 131992..."
100,1015,1,"[251689, 66334, 203557, 213713, 72522, 30051, ...","[64550, 203557, 16769, 54047, 68844, 140314, 2..."
101,1036,2,"[120390, 50662, 155673, 5715, 152397, 250011, ...","[86790, 221079, 235367, 129996, 49453, 58505, ..."
102,1040,1,"[24962, 119667, 229970, 162730, 192352, 66050,...","[180201, 174692, 202106, 29246, 79687, 191313,..."


In [131]:
tracks['track_popularity'] = tracks['track_popularity'] / 100
tracks['track_idx'] = tracks['track_idx'].astype(int)
tracks = tracks.set_index('track_idx')
playlist['popularity_mean'] = playlist['popularity_mean'] / 100
playlist['playlist_idx'] = playlist['playlist_idx'].astype(int)
playlist = playlist.set_index('playlist_idx')

In [132]:
tracks_relevant_columns = tracks[['track_popularity',
                                  'Early Years', 'Classic Era', 'Golden Era', '2000s', 'Modern Era',
                                  'Short', 'Medium', 'Long',
                                  "joy", "calm", "sadness", "fear", "energizing", "dreamy",
                                  "Instrumental / Ambient Sounds", "Soft Acoustic / Classical", "Orchestral / Soundtrack", "Mid-tempo Pop / Indie", "Upbeat Electronic / Dance", "Slow & Melancholic (Sad Songs)", "Experimental / Jazz Fusion", "Lo-Fi / Chill Vibes"]]

In [133]:
playlist['sentiment_centroid'] = playlist['sentiment_centroid'].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))
playlist['genre_centroid'] = playlist['genre_centroid'].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))

# Unpack the column into separate columns
playlist[['genre1', 'genre2', 'genre3', 'genre4', 'genre5', 'genre6', 'genre7', 'genre8']] = pd.DataFrame(playlist['genre_centroid'].tolist())

# Drop the original column if you no longer need it
playlist = playlist.drop(columns=['genre_centroid'])

# Unpack the column into separate columns
playlist[['sent1', 'sent2', 'sent3', 'sent4', 'sent5', 'sent6']] = pd.DataFrame(playlist['sentiment_centroid'].tolist())

# Drop the original column if you no longer need it
playlist = playlist.drop(columns=['sentiment_centroid'])

print(playlist)

              cluster dataset_type  num_tracks  \
playlist_idx                                     
1                   1        train          32   
2                   3         test          10   
3                   3        train          52   
4                   0          val          64   
5                   0        final          73   
...               ...          ...         ...   
1061                1        train          32   
1062                1        train          99   
1063                2        train          40   
1064                2        train          13   
1065                3        train          53   

                                                 track_idx_list  \
playlist_idx                                                      
1             ['3689' '207774' '194775' '135193' '218011' '3...   
2             ['160375' '131195' '164629' '147280' '193891' ...   
3             ['104436' '229428' '25968' '186871' '81592' '1...   
4             

In [134]:
playlist_relevant_columns = playlist[['popularity_mean', 
                                      'era_early_years_proportion', 'era_classic_era_proportion', 'era_golden_era_proportion', 'era_2000s_proportion', 'era_modern_era_proportion',
                                      'length_short_proportion', 'length_medium_proportion', 'length_long_proportion',
                                      'sent1', 'sent2', 'sent3', 'sent4', 'sent5', 'sent6',
                                      'genre1', 'genre2', 'genre3', 'genre4', 'genre5', 'genre6', 'genre7', 'genre8']]

# Relevance

In [135]:
from sklearn.metrics.pairwise import cosine_similarity

def avg_cosine_similarity(set_a, set_b, tracks_df, track_feat_columns):
    """
    Computes the average cosine similarity between all track pairs (a, b)
    where a ∈ set_a and b ∈ set_b, based on the provided track feature columns.
    """
    # Convert set_a and set_b to list if they are not already
    set_a = list(set_a)
    set_b = list(set_b)
    
    # Make sure that the track indices are valid
    a_vectors = tracks_df.loc[set_a, track_feat_columns].values
    b_vectors = tracks_df.loc[set_b, track_feat_columns].values
    
    if a_vectors.shape[0] == 0 or b_vectors.shape[0] == 0:
        return 0.0  # Avoid similarity calculation if vectors are empty

    # Compute the cosine similarity between all pairs of tracks in a_vectors and b_vectors
    sim_matrix = cosine_similarity(a_vectors, b_vectors)
    return sim_matrix.mean()

# Example: To apply this on each playlist
def compute_relevance_for_playlist(row, tracks_df, track_feat_columns):
    true_tracks = row['tracks_to_predict']
    recommended_tracks = row['recommended_tracks']
    
    # Calculate cosine similarity for the top k recommended tracks and true tracks
    sim_score = avg_cosine_similarity(recommended_tracks, true_tracks, tracks_df, track_feat_columns)
    return sim_score

# Apply this to the dataset and create a new column in the `merged` DataFrame
merged['relevance'] = merged.apply(compute_relevance_for_playlist, axis=1, 
                                               tracks_df=tracks, track_feat_columns=tracks_relevant_columns.columns)

# Print the updated DataFrame
print(merged[['playlist_idx', 'relevance']].head())

   playlist_idx  relevance
0             2   0.451900
1             7   0.482610
2            10   0.711988
3            17   0.526658
4            28   0.666918


In [136]:
# Group by the 'cluster' column and calculate the average cosine similarity for each cluster
cluster_avg_relevance = merged.groupby('cluster')['relevance'].mean()

# Print the average cosine similarity for each cluster
print(cluster_avg_relevance)

cluster
0    0.655609
1    0.617809
2    0.637795
3    0.516719
Name: relevance, dtype: float64


# Diversity

In [137]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def calculate_diversity_score(recommended_tracks, tracks_df, track_feat_columns):
    """
    Calculate the diversity score for a given set of recommended tracks.
    The diversity score is based on the average pairwise cosine similarity between recommended tracks.
    A lower cosine similarity indicates higher diversity.
    """
    # Extract the feature vectors of the recommended tracks
    recommended_vectors = tracks_df.loc[recommended_tracks, track_feat_columns].values
    
    if recommended_vectors.shape[0] == 0:
        return 0.0  # If there are no recommended tracks, return a default score
    
    # Compute the pairwise cosine similarity between all recommended tracks
    sim_matrix = cosine_similarity(recommended_vectors)
    
    # We remove the diagonal of the similarity matrix, as it represents self-similarity
    np.fill_diagonal(sim_matrix, 0)  # Set the diagonal to 0 to ignore self-similarity
    
    # Calculate the average of the off-diagonal similarities
    avg_similarity = sim_matrix.sum() / (sim_matrix.shape[0] * (sim_matrix.shape[0] - 1))
    
    # Diversity score: lower cosine similarity means higher diversity
    diversity_score = 1 - avg_similarity  # You can also directly return avg_similarity if you prefer
    
    return diversity_score

# Example: Apply this to each playlist
def compute_diversity_for_playlist(row, tracks_df, track_feat_columns):
    recommended_tracks = row['recommended_tracks']
    
    # Calculate the diversity score for the recommended tracks
    diversity_score = calculate_diversity_score(recommended_tracks, tracks_df, track_feat_columns)
    return diversity_score

# Apply this function to the merged DataFrame to get the diversity score for each playlist
merged['diversity'] = merged.apply(compute_diversity_for_playlist, axis=1, 
                                          tracks_df=tracks, track_feat_columns=tracks_relevant_columns.columns)

# Print the dataframe with the diversity scores
print(merged[['playlist_idx', 'diversity']].head())

   playlist_idx  diversity
0             2   0.341918
1             7   0.482272
2            10   0.292408
3            17   0.333999
4            28   0.320397


In [138]:
# Group by the 'cluster' column and calculate the average cosine similarity for each cluster
cluster_avg_diversity = merged.groupby('cluster')['diversity'].mean()

# Print the average cosine similarity for each cluster
print(cluster_avg_diversity)

cluster
0    0.313249
1    0.350581
2    0.329016
3    0.432466
Name: diversity, dtype: float64


# Serendipity

In [139]:
# After loading and preprocessing your playlist DataFrame
nan_counts = playlist.isna().sum()
print("NaN counts per column:")
print(nan_counts[nan_counts > 0])

NaN counts per column:
genre1    63
genre2    63
genre3    63
genre4    63
genre5    63
genre6    63
genre7    63
genre8    63
sent1     63
sent2     63
sent3     63
sent4     63
sent5     63
sent6     63
dtype: int64


In [140]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def compute_novelty(row, playlist_relevant_df, tracks_df, playlist_feat_columns, track_feat_columns):
    # Get playlist feature vector (assume index of row matches that in playlist_relevant_df)
    playlist_vector = playlist_relevant_df.loc[row.name, playlist_feat_columns].values.reshape(1, -1)

    # Get track IDs for recommended tracks
    recommended_ids = row['recommended_tracks']
    
    # Make sure it's a list of valid indices
    recommended_ids = list(recommended_ids)
    
    # Get feature vectors for recommended tracks
    track_vectors = tracks_df.loc[recommended_ids, track_feat_columns].values
    
    if track_vectors.shape[0] == 0:
        return 0.0  # Avoid errors on empty recommended lists
    
    # Compute cosine similarity between playlist and each recommended track
    sim_scores = cosine_similarity(playlist_vector, track_vectors)[0]  # Shape (n_tracks,)
    
    # Novelty = 1 - average similarity
    novelty_score = 1 - np.mean(sim_scores)
    return novelty_score


# Align indices
merged = merged.reset_index(drop=True)
playlist = playlist.reset_index(drop=True)

merged['novelty'] = merged.apply(
    compute_novelty,
    axis=1,
    playlist_relevant_df=playlist,
    tracks_df=tracks,
    playlist_feat_columns=playlist_relevant_columns.columns,
    track_feat_columns=tracks_relevant_columns.columns
)

merged['serendipity'] = merged['relevance'] * merged['novelty']

print(merged[['playlist_idx', 'serendipity']].head())

   playlist_idx  serendipity
0             2     0.280629
1             7     0.224418
2            10     0.337659
3            17     0.110593
4            28     0.136389


In [141]:
# Group by the 'cluster' column and calculate the average cosine similarity for each cluster
cluster_avg_serendipity = merged.groupby('cluster')['serendipity'].mean()

# Print the average cosine similarity for each cluster
print(cluster_avg_serendipity)

cluster
0    0.251540
1    0.176079
2    0.165605
3    0.199371
Name: serendipity, dtype: float64
